In [1]:
from ipynb.fs.defs.agent import DQNAgent
from ipynb.fs.defs.trading_env import TradingEnv
from ipynb.fs.defs.data import reinforcement_data
from ipynb.fs.defs.features import build_features
from ipynb.fs.defs.data import reinforcement_data
   
from ipynb.fs.defs.save_load import save_model
from ipynb.fs.defs.save_load import load_model
import random

In [2]:
stocks = [
    "AAPL",
    "MSFT",
    "GOOG",
    "AMZN",
    "META",
    "INTC",
    "PYPL",
    "BABA",
    "NKE",
    "DIS",
    "IBM",
    "T",
    "PFE"
]

feature_cache = {}

for ticker in stocks:

    try:

        prices = reinforcement_data(ticker,"2020-01-01","2024-01-01").squeeze()

        feature_cache[ticker] =  build_features(prices)

        print(f"Loaded {ticker}")

    except Exception as e:

        print(f"Skipping {ticker}: {e}")

[*********************100%***********************]  1 of 1 completed


Loaded AAPL


[*********************100%***********************]  1 of 1 completed


Loaded MSFT


[*********************100%***********************]  1 of 1 completed


...

Loaded IBM


[*********************100%***********************]  1 of 1 completed


Loaded T


[*********************100%***********************]  1 of 1 completed

Loaded PFE


In [3]:
agent = DQNAgent()
episodes = 500

for episode in range(episodes):

    ticker = random.choice(stocks)

    env = TradingEnv(feature_cache[ticker])

    state = env.reset()
    done = False

    episode_reward = 0

    while not done:

        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)

        agent.replay_buffer.push(
            (
                state,
                action,
                reward,
                next_state,
                done
            )
        )

        agent.train_step()

        state = next_state

        episode_reward += reward

    agent.epsilon = max(agent.epsilon_min,agent.epsilon* agent.epsilon_decay)

    agent.update_target_net()

    print(
        f"Episode {episode+1}/{episodes} "
        f"| Reward: {episode_reward:.4f} "
        f"| Epsilon: {agent.epsilon:.4f}"
    )

Episode 1/500 | Reward: 0.3965 | Epsilon: 0.9900
Episode 2/500 | Reward: -0.5385 | Epsilon: 0.9801
Episode 3/500 | Reward: 0.1304 | Epsilon: 0.9703
Episode 4/500 | Reward: 0.4056 | Epsilon: 0.9606
Episode 5/500 | Reward: -0.4836 | Epsilon: 0.9510
Episode 6/500 | Reward: 0.3755 | Epsilon: 0.9415
Episode 7/500 | Reward: -0.0544 | Epsilon: 0.9321
Episode 8/500 | Reward: 0.4437 | Epsilon: 0.9227
Episode 9/500 | Reward: -0.1255 | Epsilon: 0.9135
Episode 10/500 | Reward: -0.2211 | Epsilon: 0.9044
Episode 11/500 | Reward: 0.1057 | Epsilon: 0.8953
Episode 12/500 | Reward: 0.1831 | Epsilon: 0.8864
Episode 13/500 | Reward: 0.3686 | Epsilon: 0.8775
Episode 14/500 | Reward: 0.4768 | Epsilon: 0.8687
Episode 15/500 | Reward: -0.2950 | Epsilon: 0.8601
Episode 16/500 | Reward: 0.9999 | Epsilon: 0.8515
Episode 17/500 | Reward: 0.4875 | Epsilon: 0.8429
Episode 18/500 | Reward: 0.3563 | Epsilon: 0.8345
Episode 19/500 | Reward: 0.1750 | Epsilon: 0.8262
Episode 20/500 | Reward: -0.1639 | Epsilon: 0.8179
Ep

In [4]:
save_model(agent,filepath="models/dqn_trader.pt",episodes=episodes)

Model saved to models/dqn_trader.pt
